In [6]:
import pandas as pd
import plotly.express as px
from pathlib import Path
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import adjusted_rand_score

proc_dir = Path("../data/processed")
threshold = 2

In [7]:
emb = pd.read_csv(proc_dir / "adr_embeddings.csv")
clusters = pd.read_csv(proc_dir / f"adr_clusters_cutoff_{threshold}.csv")

df = emb.merge(clusters, on="adr", how="left")
print(f"Embeddings: {len(df)} ADRs | Clusters: {df['cluster'].nunique()}")
df.head()

Embeddings: 58 ADRs | Clusters: 7


,adr,x,y,cluster,degree
0,anxiety,-0.993575,5.342188,3.0,16.0
1,withdrawal,-3.319052,1.038468,1.0,15.0
2,insomnia,-4.844678,3.180313,4.0,9.0
3,withdrawals,-1.547351,7.389265,3.0,10.0
4,depression,-2.209633,4.250505,1.0,8.0


In [8]:
meddra_hierarchy = {
    "Psychiatric disorders": {
        "Mood and anxiety disorders": [
            "depression", "depressed", "anxiety", "anxious", "panic attack",
            "panic attacks", "irritable", "tolerance", "addicted", "addiction",
            "addictive", "withdrawal", "withdrawals"
        ],
        "Psychotic disorders": ["paranoia", "mania", "psychosis", "manic"]
    },
    "Nervous system disorders": {
        "Sleep disorders": ["insomnia", "nightmares", "fatigue", "exhausted", "groggy"],
        "Neurological symptoms": [
            "tremors", "tinnitus", "migraines", "headaches", "numb", "derealization"
        ]
    },
    "Gastrointestinal disorders": {
        "GI symptoms": ["nausea", "vomiting", "diarrhea", "cravings"]
    },
    "Metabolism and nutrition disorders": {
        "Appetite and weight": ["weight gain", "dehydrated"]
    },
    "General disorders and administration site conditions": {
        "Systemic responses": ["sweating", "keyed-up"]
    }
}

In [10]:
# flattening hierarchy for easier maping

records = []
for soc, hlt_dict in meddra_hierarchy.items():
    for hlt, terms in hlt_dict.items():
        for term in terms:
            records.append({"adr": term, "hlt": hlt, "soc": soc})

meddra_df = pd.DataFrame(records)
print(f"MedDRA mapping created for {len(meddra_df)} ADRs.")

MedDRA mapping created for 36 ADRs.


## Merging mapping with clusters

In [11]:
clusters = pd.read_csv(proc_dir / f"adr_clusters_cutoff_{threshold}.csv")
df = clusters.merge(meddra_df, on="adr", how="left")
df["soc"].fillna("Unclassified", inplace=True)
df["hlt"].fillna("Unclassified", inplace=True)

/var/folders/yz/bx5g_gys4mz1tzj0vc3sk7km0000gn/T/ipykernel_4823/2688781351.py:3: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df["soc"].fillna("Unclassified", inplace=True)
/var/folders/yz/bx5g_gys4mz1tzj0vc3sk7km0000gn/T/ipykernel_4823/2688781351.py:4: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always b

## Plot

In [12]:
emb = pd.read_csv(proc_dir / "adr_embeddings.csv")
df = emb.merge(df, on="adr", how="left")

fig = px.scatter(
    df, x="x", y="y",
    color="soc",
    hover_data=["adr", "cluster", "hlt"],
    title=f"ADR Embedding Space (Node2Vec t-SNE | Edges ≥ {threshold})",
    color_discrete_sequence=px.colors.qualitative.Bold
)
fig.update_traces(marker=dict(size=10, opacity=0.8))
fig.show()
fig.write_html(proc_dir / f"fig_embeddings_soc_cutoff_{threshold}.html")